# SL-7 Final Holdout v16

**Frozen final experiment.** Seven pre-registered blind-spot hypotheses, 20 calibration items, and 150 scored multimodal items.

Do not edit prompts, answer keys, thresholds, or images after inspecting model outputs.

In [1]:
!pip install -q -U "transformers==5.17.0" "accelerate>=1.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 95.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 18.6 MB/s eta 0:00:00


In [2]:
import os,glob,csv,json,time,platform,sys,zipfile,importlib,random
from pathlib import Path
from PIL import Image
import torch, transformers
MODEL_ID="Qwen/Qwen3-VL-4B-Instruct"
MAX_NEW_TOKENS=64
INFERENCE_SEED=20261004
random.seed(INFERENCE_SEED); torch.manual_seed(INFERENCE_SEED)

hits=[]
for p in glob.glob("/kaggle/input/**/dataset/items.csv",recursive=True):
    root=Path(p).parent.parent
    if (root/"VERSION_SL7_v16.txt").exists() and (root/"eval/score.py").exists(): hits.append(root)
if not hits: raise FileNotFoundError("Add the snakes-ladders-SL7-final-v16 package as a Kaggle input dataset.")
ROOT=hits[0]; ITEMS=list(csv.DictReader(open(ROOT/"dataset/items.csv",encoding="utf-8")))
CAL=[x for x in ITEMS if x["part"]=="CAL"]; SCORED=[x for x in ITEMS if x["part"]=="SCORED"]
print("Project root:",ROOT); print("Calibration:",len(CAL),"Scored:",len(SCORED))
OUT=Path("/kaggle/working/results_SL7_final_v16"); OUT.mkdir(exist_ok=True)
sys.path.insert(0,str(ROOT/"eval")); import score; importlib.reload(score)

Project root: /kaggle/input/datasets/sabahnushra/snak-ladder-redesign/snakes-ladders-SL7-final-v16
Calibration: 20 Scored: 150


<module 'score' from '/kaggle/input/datasets/sabahnushra/snak-ladder-redesign/snakes-ladders-SL7-final-v16/eval/score.py'>

In [3]:
try:
    from transformers import Qwen3VLForConditionalGeneration as ModelClass
except ImportError:
    from transformers import AutoModelForImageTextToText as ModelClass
from transformers import AutoProcessor
if not torch.cuda.is_available(): raise RuntimeError("Enable a Kaggle GPU accelerator.")
try:
    model=ModelClass.from_pretrained(MODEL_ID,dtype=torch.float16,device_map={"":0},attn_implementation="sdpa")
except TypeError:
    model=ModelClass.from_pretrained(MODEL_ID,torch_dtype=torch.float16,device_map={"":0},attn_implementation="sdpa")
model.eval(); processor=AutoProcessor.from_pretrained(MODEL_ID)
env={"model_id":MODEL_ID,"parameters":sum(p.numel() for p in model.parameters()),"transformers":transformers.__version__,"torch":torch.__version__,"cuda":torch.version.cuda,"gpu":torch.cuda.get_device_name(0),"python":platform.python_version(),"dtype":"float16","attention":"sdpa","decoding":"greedy","max_new_tokens":MAX_NEW_TOKENS,"inference_seed":INFERENCE_SEED,"generation_seed":20261004}
(OUT/"environment.json").write_text(json.dumps(env,indent=2)); print(json.dumps(env,indent=2))

config.json:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/64.7k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

{
  "model_id": "Qwen/Qwen3-VL-4B-Instruct",
  "parameters": 4437815808,
  "transformers": "5.17.0",
  "torch": "2.11.0+cu128",
  "cuda": "12.8",
  "gpu": "Tesla T4",
  "python": "3.13.15",
  "dtype": "float16",
  "attention": "sdpa",
  "decoding": "greedy",
  "max_new_tokens": 64,
  "inference_seed": 20261004,
  "generation_seed": 20261004
}


In [4]:
RAW=OUT/"raw_outputs.csv"
outrows=[]
def run_item(it):
    images=[]; content=[]
    for rel in [p for p in it["image_paths"].split("|") if p]:
        with Image.open(ROOT/rel) as im: images.append(im.convert("RGB").copy())
        content.append({"type":"image"})
    content.append({"type":"text","text":it["prompt"]})
    messages=[{"role":"user","content":content}]
    prompt=processor.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    inputs=processor(text=[prompt],images=images,return_tensors="pt").to(model.device)
    if "pixel_values" in inputs: inputs["pixel_values"]=inputs["pixel_values"].to(torch.float16)
    t0=time.time()
    with torch.inference_mode(): generated=model.generate(**inputs,max_new_tokens=MAX_NEW_TOKENS,do_sample=False)
    new=generated[:,inputs["input_ids"].shape[1]:]
    reply=processor.batch_decode(new,skip_special_tokens=True,clean_up_tokenization_spaces=False)[0].strip()
    return {"item_id":it["item_id"],"raw_output":reply,"seconds":round(time.time()-t0,2)}

def save_raw():
    with open(RAW,"w",newline="",encoding="utf-8") as f:
        w=csv.DictWriter(f,fieldnames=["item_id","raw_output","seconds"]); w.writeheader(); w.writerows(outrows)

# CALIBRATION FIRST
for idx,it in enumerate(CAL,1):
    outrows.append(run_item(it)); save_raw(); print(f"CAL {idx}/{len(CAL)}",it["item_id"])

# Check frozen visual-legibility gates before the scored run.
pred={r["item_id"]:r["raw_output"] for r in outrows}
groups={}
for it in CAL:
    parsed=score.parse_strict(pred[it["item_id"]],it["answer_format"]); ok=parsed.lower()==it["correct_answer"].lower() if parsed else False
    groups.setdefault(it["task_family"],[]).append(ok)
print({k:(sum(v),len(v)) for k,v in groups.items()})
requirements={"CAL_number":7,"CAL_endpoint":3,"CAL_die":4,"CAL_multi_image":3}
failed=[k for k,need in requirements.items() if sum(groups.get(k,[]))<need]
if failed:
    raise RuntimeError(f"Calibration gate failed for {failed}. Do NOT run the scored holdout until visual legibility is fixed without looking at scored outputs.")
print("Calibration gate PASSED. Starting frozen scored holdout.")

CAL 1/20 CAL_NUM_01
CAL 2/20 CAL_NUM_02
CAL 3/20 CAL_NUM_03
CAL 4/20 CAL_NUM_04
CAL 5/20 CAL_NUM_05
CAL 6/20 CAL_NUM_06
CAL 7/20 CAL_NUM_07
CAL 8/20 CAL_NUM_08
CAL 9/20 CAL_END_01
CAL 10/20 CAL_END_02
CAL 11/20 CAL_END_03
CAL 12/20 CAL_END_04
CAL 13/20 CAL_DIE_01
CAL 14/20 CAL_DIE_02
CAL 15/20 CAL_DIE_03
CAL 16/20 CAL_DIE_04
CAL 17/20 CAL_CMP_01
CAL 18/20 CAL_CMP_02
CAL 19/20 CAL_CMP_03
CAL 20/20 CAL_CMP_04
{'CAL_number': (8, 8), 'CAL_endpoint': (3, 4), 'CAL_die': (4, 4), 'CAL_multi_image': (3, 4)}
Calibration gate PASSED. Starting frozen scored holdout.


In [5]:
for idx,it in enumerate(SCORED,1):
    outrows.append(run_item(it)); save_raw()
    if idx%10==0 or idx==len(SCORED): print(f"SCORED {idx}/{len(SCORED)} complete")
print("Raw outputs saved:",RAW)

SCORED 10/150 complete
SCORED 20/150 complete
SCORED 30/150 complete
SCORED 40/150 complete
SCORED 50/150 complete
SCORED 60/150 complete
SCORED 70/150 complete
SCORED 80/150 complete
SCORED 90/150 complete
SCORED 100/150 complete
SCORED 110/150 complete
SCORED 120/150 complete
SCORED 130/150 complete
SCORED 140/150 complete
SCORED 150/150 complete
Raw outputs saved: /kaggle/working/results_SL7_final_v16/raw_outputs.csv


In [6]:
scored_rows,family_rows,cal_pass=score.score(ROOT/"dataset/items.csv",RAW,OUT)
print((OUT/"summary.md").read_text())

# SL-7 Final Holdout Summary

Total scored: 150 | Calibration: 20

## Calibration gates

| Calibration | Correct | Gate |
|---|---:|---|
| number | 8/8 | PASS |
| endpoint | 3/4 | PASS |
| die | 4/4 | PASS |
| multi_image | 3/4 | PASS |

**Overall calibration gate:** PASS

## Seven frozen hypotheses

| Family | Control/component | Probe | Gap | Signature | Confirmed | 95% CI control | 95% CI probe |
|---|---:|---:|---:|---:|---|---|---|
| B1_zigzag_direction | 10/10 | 9/10 | 10.0 pp | 1/1 | **NO** | 72.2–100.0% | 59.6–98.2% |
| B2_hidden_reconstruction | 10/10 | 1/10 | 90.0 pp | 3/9 | **NO** | 72.2–100.0% | 1.8–40.4% |
| B3_endpoint_anchoring | 10/10 | 0/10 | 100.0 pp | 10/10 | **YES** | 72.2–100.0% | 0.0–27.8% |
| B4_perception_to_action | 5/10 | 1/5 | 30.0 pp | 1/4 | **NO** | 23.7–76.3% | 3.6–62.4% |
| B5_multi_image_position | 3/4 | 9/16 | 18.8 pp | N/A | **NO** | 30.1–95.4% | 33.2–76.9% |
| B6_conditioned_counting | 8/10 | 0/10 | 80.0 pp | 6/10 | **YES** | 49.0–94.3% | 0.0–27.8% |


In [7]:
zip_path=Path("/kaggle/working/results_SL7_final_v16.zip")
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file(): z.write(p,arcname=p.name)
print("Download:",zip_path)

Download: /kaggle/working/results_SL7_final_v16.zip
